In [80]:
import numpy as np
import random as rand
import matplotlib.pyplot as plt
import itertools
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import os
import json
from ELO import Elo
import joblib

In [79]:
def load_training_data(scenario, split):
    base_path = "../Simulated_Data"
    
    filename = f"scenario_{scenario}_{split}_data.csv"
    filepath = os.path.join(base_path, filename)
    
    return pd.read_csv(filepath)

In [78]:
import numpy as np

def construct_data(x, initial_id, last_id, num_tasks, num_skills):
    """
    Converts raw student interaction data into structured input features
    and target labels.

    Feature layout per row:
    [s0, s0_past_success, s0_past_fail,
     s1, s1_past_success, s1_past_fail,
     ...
     s_{num_skills-1}, ...,
     task_0, ..., task_(num_tasks-1)]
    """
    x_numpy = x.to_numpy()

    x_data = []
    y_data = []

    for student_id in range(initial_id, last_id):
        student_indices = np.where(x_numpy[:, 0] == student_id)[0]

        previous_time = -1
        past_success = np.zeros(num_skills)
        past_fail = np.zeros(num_skills)

        for idx in student_indices:
            correct = int(x_numpy[idx, 3])
            skill = int(x_numpy[idx, 2])
            task = int(x_numpy[idx, 1])
            time = int(x_numpy[idx, -1])

            if time != previous_time:
                feature_vector = [0] * (3 * num_skills + num_tasks)

            feature_vector[3 * num_skills + task] = 1

            skill_offset = 3 * skill
            feature_vector[skill_offset] = 1
            feature_vector[skill_offset + 1] = past_success[skill]
            feature_vector[skill_offset + 2] = past_fail[skill]

            if correct:
                past_success[skill] += 1
            else:
                past_fail[skill] += 1

            if x_data and time == previous_time:
                x_data[-1] = feature_vector
                y_data[-1] = correct
            else:
                x_data.append(feature_vector)
                y_data.append(correct)

            previous_time = time

    return x_data, y_data

In [4]:
def save_model(model, scenario_id, base_dir=None):
    """
    Save a trained model checkpoint for a given scenario.

    Args:
        model       : the trained PyTorch model
        scenario_id : scenario identifier (e.g. 1–6, or a string name)
        learning_rate, hidden_size, batch_size : hyperparameters to store
        base_dir    : optional override for the Trained_Models folder path
    """
    if base_dir is None:
        base_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'Trained_Models'))

    os.makedirs(base_dir, exist_ok=True)

    model_path = os.path.join(base_dir, f'PFA_model_scenario_{scenario_id}.pkl')
    joblib.dump(model, model_path)

    print(f"[Scenario {scenario_id}] Model saved → {model_path}")
    return model_path

# Start Training

In [81]:
with open('../elo_variable.json', 'r') as Elo_Data:
    elo_data = json.load(Elo_Data)

print(elo_data)

{'globals': {'students': 500, 'init_skill_level': [0.0, 0.0], 'k_success': 1, 'k_fail': 0.5}, 'scenarios': [{'id': 1, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 0], [0, 1]], 'difficulty_level': [1, 1], 'depends': [-1, -1]}, {'id': 2, 'num_tasks': 2, 'num_skills': 2, 'q_matrix': [[1, 1], [1, 1]], 'difficulty_level': [0, 1], 'depends': [-1, -1]}, {'id': 3, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0], [0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1], 'depends': [-1, -1, -1, -1]}, {'id': 4, 'num_tasks': 4, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1], [0, 1, 1, 1]], 'difficulty_level': [1, 1, 0, 1], 'depends': [-1, -1, -1, -1]}, {'id': 5, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 1, 0, 1], [0, 1, 1, 0, 1, 1]], 'difficulty_level': [0, 0, 0, 1, 1, 1], 'depends': [-1, -1, -1, -1, -1, -1]}, {'id': 6, 'num_tasks': 6, 'num_skills': 2, 'q_matrix': [[1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1]], 'difficulty_level': [0, 0, 1, 1, 2, 2], 'depends': [-1, -1, -1, -1, -1, -1]}

In [82]:
all_data = {}

level_skill   = [] 
mastery_level = 1.5
i = 0

global_values = elo_data["globals"]
scenarios = elo_data["scenarios"]

students          = global_values["students"]
init_skill_level  = np.array(global_values["init_skill_level"])
k_success         = global_values["k_success"]
k_fail            = global_values["k_fail"]

In [ ]:
for scenario in scenarios:
    scenario_id    = scenario["id"]
    num_tasks      = scenario["num_tasks"]
    difficulty_level = np.array(scenario["difficulty_level"])
    q_matrix       = np.array(scenario["q_matrix"])
    depends = np.array(scenario["depends"])
    results = [] 
    results_mod = []

    df = load_training_data(scenario_id, "train")
    X_train, y_train   = construct_data(df, 0, 400, num_tasks)
    test_df = load_training_data(scenario_id, "test")
    X_test,  y_test = construct_data(test_df, 400, 449, num_tasks)

    PFA_model = LogisticRegression(max_iter=1000, C=0.1, fit_intercept=False)
    PFA_model.fit(X_train, y_train)

    # save_model(PFA_model, scenario_id)

    y_pred = PFA_model.predict(X_train)
    # Calculate the accuracy
    accuracy = accuracy_score(y_train, y_pred)
    
    print(f"Train Accuracy: {accuracy}")

    y_pred_test = PFA_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    
    print(f"Test Accuracy: {test_accuracy}")

[Scenario 1] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_1.pkl
Train Accuracy: 0.7285325414501361
Test Accuracy: 0.7239915074309978
[Scenario 2] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_2.pkl
Train Accuracy: 0.7614920874152223
Test Accuracy: 0.7612903225806451
[Scenario 3] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_3.pkl
Train Accuracy: 0.7599284436493738
Test Accuracy: 0.7521008403361344
[Scenario 4] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_4.pkl
Train Accuracy: 0.7219125843355823
Test Accuracy: 0.7050359712230215
[Scenario 5] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_5.pkl
Train Accuracy: 0.7611326786937657
Test Accuracy: 0.7532467532467533
[Scenario 6] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_6.pkl
Train Accuracy: 0.8071003488610712
Test Accuracy:

In [92]:
for scenario in scenarios:
    scenario_id    = scenario["id"]
    num_tasks      = scenario["num_tasks"]
    num_skills     = scenario["num_skills"]
    difficulty_level = np.array(scenario["difficulty_level"])
    q_matrix       = np.array(scenario["q_matrix"])
    depends = np.array(scenario["depends"])
    results = [] 
    results_mod = []

    df = load_training_data(scenario_id, "train")
    X_train, y_train   = construct_data(df, 0, 400, num_tasks, num_skills)
    test_df = load_training_data(scenario_id, "test")
    X_test,  y_test = construct_data(test_df, 400, 449, num_tasks, num_skills)

    PFA_model = LogisticRegression(max_iter=1000, C=0.1, fit_intercept=False)
    PFA_model.fit(X_train, y_train)

    save_model(PFA_model, scenario_id)

    y_pred = PFA_model.predict(X_train)
    # Calculate the accuracy
    accuracy = accuracy_score(y_train, y_pred)
    
    print(f"Train Accuracy: {accuracy}")

    y_pred_test = PFA_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    
    print(f"Test Accuracy: {test_accuracy}")

[Scenario 1] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_1.pkl
Train Accuracy: 0.6487034949267193
Test Accuracy: 0.66600790513834
[Scenario 2] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_2.pkl
Train Accuracy: 0.7062937062937062
Test Accuracy: 0.7259786476868327
[Scenario 3] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_3.pkl
Train Accuracy: 0.6726720647773279
Test Accuracy: 0.6827586206896552
[Scenario 4] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_4.pkl
Train Accuracy: 0.6566656356325394
Test Accuracy: 0.6492146596858639
[Scenario 5] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_5.pkl
Train Accuracy: 0.6803628601921025
Test Accuracy: 0.6827133479212254
[Scenario 6] Model saved → /Users/GraceCeline/Bachelorarbeit/Trained_Models/PFA_model_scenario_6.pkl
Train Accuracy: 0.7093681917211329
Test Accuracy: 0